Bullet-point instructions for “future you” to sanitize

A. Build an index first (don’t start with model code)

Create a table sessions_index with one row per (year, round, session).

Columns: year, round, event_name, country/location, session (“Q”), cache_path (optional), load_status, n_laps, n_drivers, has_results, has_weather, has_trackstatus.

Save it as CSV. This becomes your “truth”.

B. Session-level eligibility rules (exclude whole sessions early)

Skip non-race weekends (testing, track sessions). Rule: must be a “Grand Prix” weekend or an “EventFormat” that includes Qualifying.

Require minimum coverage:

≥18 unique drivers with non-null LapTime

results table exists and has positions for those drivers

Require non-empty laps after filtering invalid laps (see below).

If any of the required components are missing (e.g., results/positions), mark usable=False with a reason.

C. Driver/lap sanitation (per session)

Start from laps = ses.laps.

Keep only laps with LapTime not null.

Remove clearly invalid laps:

Deleted == True (and optionally keep DeletedReason for audit)

optionally: if IsAccurate == False, either drop or keep but flag (decide once, consistently)

For each driver, pick best lap as min(LapTime) after the filters.

Confirm each driver has Sector1/2/3 times if you intend to use them; otherwise set to NaN consistently.

D. Results and labels (avoid mixing “grid” vs “quali classification”)

Merge best laps with ses.results.

Use “qualifying classification position” (the session result position) as your label, not race grid position.

Drop drivers where position is missing / non-numeric.

If a driver is DSQ/DNS and that yields missing/odd position, drop them (log it).

E. Feature engineering hygiene (consistency over cleverness)

Convert all timedeltas to seconds with one helper function.

Keep a fixed feature schema across all years:

If a column doesn’t exist for some year, create it and fill NaN.

Store categorical encoding decisions:

Prefer stable IDs (DriverNumber/Abbreviation + year) over pandas category codes that can reshuffle.

If you must encode, persist the mapping dict.

F. Quality checks you run every time (fast, automated)

For each session, assert:

18–20 drivers present after filtering

no duplicate drivers

positions are 1..20-ish without wild gaps (some gaps ok, but log)

Dataset-level checks:

counts by year and circuit

missingness rates per feature

distribution of LapTime / sector times for obvious outliers


In [ ]:
# Bullet-point instructions for “future you” to sanitize

# A. Build an index first (don’t start with model code)

# Create a table sessions_index with one row per (year, round, session).

# Columns: year, round, event_name, country/location, session (“Q”), cache_path (optional), load_status, n_laps, n_drivers, has_results, has_weather, has_trackstatus.

# Save it as CSV. This becomes your “truth”.

# B. Session-level eligibility rules (exclude whole sessions early)

# Skip non-race weekends (testing, track sessions). Rule: must be a “Grand Prix” weekend or an “EventFormat” that includes Qualifying.

# Require minimum coverage:

# ≥18 unique drivers with non-null LapTime

# results table exists and has positions for those drivers

# Require non-empty laps after filtering invalid laps (see below).

# If any of the required components are missing (e.g., results/positions), mark usable=False with a reason.

# C. Driver/lap sanitation (per session)

# Start from laps = ses.laps.

# Keep only laps with LapTime not null.

# Remove clearly invalid laps:

# Deleted == True (and optionally keep DeletedReason for audit)

# optionally: if IsAccurate == False, either drop or keep but flag (decide once, consistently)

# For each driver, pick best lap as min(LapTime) after the filters.

# Confirm each driver has Sector1/2/3 times if you intend to use them; otherwise set to NaN consistently.

# D. Results and labels (avoid mixing “grid” vs “quali classification”)

# Merge best laps with ses.results.

# Use “qualifying classification position” (the session result position) as your label, not race grid position.

# Drop drivers where position is missing / non-numeric.

# If a driver is DSQ/DNS and that yields missing/odd position, drop them (log it).

# E. Feature engineering hygiene (consistency over cleverness)

# Convert all timedeltas to seconds with one helper function.

# Keep a fixed feature schema across all years:

# If a column doesn’t exist for some year, create it and fill NaN.

# Store categorical encoding decisions:

# Prefer stable IDs (DriverNumber/Abbreviation + year) over pandas category codes that can reshuffle.

# If you must encode, persist the mapping dict.

# F. Quality checks you run every time (fast, automated)

# For each session, assert:

# 18–20 drivers present after filtering

# no duplicate drivers

# positions are 1..20-ish without wild gaps (some gaps ok, but log)

# Dataset-level checks:

# counts by year and circuit

# missingness rates per feature

# distribution of LapTime / sector times for obvious outliers